# 98장 · 정규표현식 치환과 경계 조건

정규표현식으로 표기 차이를 정리할 때는 일치 범위를 제한해야 합니다. 바꿔야 할 이름과 접두어만 같은 다른 이름을 함께 시험합니다.

‘장비A’는 ‘장비 A’와 같은 자료를 찾아야 하지만 ‘장비AB’는 그대로 두어야 합니다. 문서와 검색 규칙을 고정하고 장비명 정규화의 영향만 측정합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## re.sub()로 장비명 공백 정규화

실험 카드　고정 항목 rules-v1, 공개 개발 질문·기대 근거, 점검 규칙 / 변경 항목 장비 A·B 표기 정리 / 기대 Q98-N01이 장비 A로 인식됨. 가설이 틀릴 수도 있으니 공백 정리 전의 질문을 보관합니다. 사용자가 무엇을 썼는지 복구할 수 있어야 합니다.


### 셀 01 · re.sub()의 장비명 정규화

끝 또는 다음 문자가 영문·숫자가 아닌 위치에서만 한 글자 장비명을 인정합니다.


In [1]:
import re

def normalize_question(question):
    return re.sub(r"장비([AB])(?=$|[^A-Za-z0-9])", r"장비 \1", question)

original = "장비A의 점검 전 확인 사항은?"
normalized = normalize_question(original)
print(original)
print(normalized)

장비A의 점검 전 확인 사항은?
장비 A의 점검 전 확인 사항은?


둘째 줄에는 장비 A 사이의 공백이 생깁니다. ‘의’ 뒤의 질문 내용은 같습니다. 원문과 처리본을 함께 두면 답이 바뀐 원인을 설명할 수 있습니다.

‘장비A’는 ‘장비 A’로 고쳐도 같은 대상을 가리킵니다. 반면 ‘장비AB’까지 일괄 치환하면 등록되지 않은 이름 일부에 규칙이 적용됩니다. 수정된 두 표기와 그대로 둬야 할 표기를 같은 테스트에 넣어야 치환 범위를 확인할 수 있습니다.


### 전방 탐색에서 다음 글자가 보존되는 이유

장비A의에서 패턴이 치환하는 부분은 장비A까지입니다. (?=...)는 뒤의 의를 읽어도 그 글자를 소비하지 않으므로 치환 뒤에도 의가 그대로 붙습니다. A 뒤에 영문 B가 이어지면 이 조건을 통과하지 않습니다.


## 접두어 오인식을 막는 반례

제공 실험 결과에서는 기본 7문항이 전후 모두 통과했고, 추가한 공개 개발 3문항 중 통과한 문항은 1개에서 3개로 늘었습니다. 팀은 이 범위에서만 수정을 채택했습니다. 두 장비 중 무엇인지 모호한 장비AB까지 잘라내는 수정은 장비를 잘못 고르는 새 오류를 만듭니다.

<!-- lesson-visual practice-space -->
**장비A와 장비AB에서 치환 위치 비교**

같은 패턴으로 네 문자열을 검사합니다. 새로 들어간 공백은 노란 칸으로, 바뀌지 않은 글자는 기존 칸으로 표시합니다.

장비A의는 장비 A의로, 끝에서 만나는 장비B는 장비 B로 바뀝니다. 다음 글자가 영문이나 숫자이면 치환하지 않습니다.

[웹 교재에서 직접 조작하기](https://langchain-lecture.vercel.app/chapter98/#practice-space)
<!-- /lesson-visual practice-space -->


#### 정규화 회귀 테스트의 판정 범위

<table style="width:100%;border-collapse:collapse"><thead><tr><th scope="col">문항 · 입력의 특징</th><th scope="col">수정 전 → 후</th><th scope="col">판정할 때 볼 점</th></tr></thead><tbody><tr><td>Q98-N01 · 장비A</td><td>실패 → 통과</td><td>장비 A로 읽어야 함</td></tr><tr><td>Q98-N02 · 장비B의 점검 전 확인 사항은?</td><td>실패 → 통과</td><td>장비 B로 읽고 RULE-B의 ‘점검 전 고정 장치를 확인합니다.’와 대조</td></tr><tr><td>Q98-G01 · 장비AB</td><td>통과 → 통과</td><td>다른 이름을 임의로 장비 A로 고치지 않음</td></tr></tbody></table><p>표는 제공된 평가 기록을 풀이한 것입니다. 셀 01과 과제 02는 입력 문자열의 변화만 직접 출력합니다.</p>


### 과제 02 · 선택 연습 · 치환 경계의 반례

Q98-G01을 같은 함수에 넣고 원문이 남는 이유를 확인하세요.

기본 7문항과 추가 3문항은 따로 셉니다. 추가 질문에서 고친 대상은 Q98-N01·N02이고 Q98-G01은 장비AB를 그대로 둘 수 있는지 보는 대조 문항입니다. 이 셀은 문자열을 확인하며, 1/3에서 3/3으로 바뀐 판정은 제공 평가 기록에서 읽습니다.


In [2]:
ambiguous = "장비AB의 점검 전 확인 사항은?"
# normalize_question에 넣고 결과를 읽으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>ambiguous = &quot;장비AB의 점검 전 확인 사항은?&quot;
print(normalize_question(ambiguous))</code></pre>
<p>실행 결과</p><pre><code>장비AB의 점검 전 확인 사항은?</code></pre>
<p>장비AB 그대로입니다. A 바로 뒤에 영문 B가 와서 공백을 넣는 조건을 충족하지 않습니다. 다른 팀은 제품명이 아니라 약어·문서 코드처럼 뜻이 갈리는 최소 반례를 고를 수 있습니다.</p>
</details>


#### 장비A의 공백을 고칠 때

<mark>더 긴 이름인 장비AB에서 같은 치환이 일어나면 다른 장비명을 바꾸게 됩니다.</mark> 수정할 입력과 보존할 입력을 쌍으로 놓으세요. 성공 예시만 늘리는 것보다 어디에서 장비명이 끝나는지 드러내는 반례 하나가 규칙의 범위를 잘 보여 줍니다.

선수 개념 · 94장 텍스트 정규화와 원문 해시

## 입력 정규화 한 줄이 바꾼 문자 범위 남기기

“장비A”를 찾도록 고친 뒤 “장비AB”까지 바뀌면 다른 장비 이름을 만들어 버립니다. 합격 문항 수 이외에 원문에서 어떤 부분이 바뀌었는지 동료가 추적할 기록을 만듭니다.

### 셀 04 · 교정 전후에서 바뀐 구간만 추출하기

모델이나 검색기를 다시 구성하지 않고 앞 셀의 normalize_question만 사용합니다. 띄어쓰기 한 칸이 실제 어느 위치에 추가됐는지 확인하세요.

In [3]:
from difflib import SequenceMatcher

before = "장비A의 점검 전 확인 사항은?"
after = normalize_question(before)
edits = [(tag, left, before[left:right], after[start:end])
         for tag, left, right, start, end in SequenceMatcher(None, before, after).get_opcodes()
         if tag != "equal"]
print(edits)
print("장비AB 보존", normalize_question(ambiguous) == ambiguous)

[('insert', 2, '', ' ')]
장비AB 보존 True


insert는 문자 위치 2에 공백 한 칸을 더했음을 뜻합니다. 장비AB는 그대로여야 합니다. 이 결과는 A·B 표기 규칙의 문자 변화만 보여 줍니다. 검색이 맞았는지는 별도로 기대 근거와 대조합니다.

### 팀 용어 경계에서 반례 고르기

팀 자료에 모델명 X와 X1이 있다면 두 이름을 따로 찾는 짝을 만드세요. 정규화 전후 문자열과 기대 문서 ID를 한 행에 써서, 이름을 표준화했지만 서로 다른 문서가 합쳐진 경우를 드러냅니다.

<mark>정규화 실험에는 바꿀 표기와 보존할 표기를 한 쌍으로 둡니다.</mark> 공백을 넣는 동작 하나가 어떠한 이름까지 바꾸는지는 문자 경계 조건으로 확인할 수 있습니다.

후보판에는 원문 질문과 정규화한 질문을 구분해 기록하고, 111장에서 같은 반례를 재실행합니다.

## 돌아보기

채택 메모에는 수정한 표현(Q98-N01·Q98-N02), 그대로 둘 표현(Q98-G01), 공개 개발 범위라는 조건을 같이 적으세요. 다음 단계는 normalization-v1이라는 이름으로 이 동작을 이어받습니다.
